# rlmflow visualization walkthrough

A guided tour of every visualization that ships with rlmflow. Everything here renders **inline in Jupyter** and runs **offline** against the saved boids trace under `examples/data/notebook-coding-agent/` — no API keys, no LLM calls.

The trace was generated by [`coding_agent.ipynb`](./coding_agent.ipynb). For the underlying `Node` query API (walk, find, where, diff, session), see [`node_basics.ipynb`](./node_basics.ipynb).

**What we cover**

1. Loading a trace
2. Inline tree rendering — `state.tree()`, Jupyter auto-render, `ascii_boxes`
3. **Interactive viewer** — `open_viewer` (Gradio: clickable graph, slider, per-agent chat)
4. Topology renders — mermaid (state / flowchart / sequence), DOT, D2
5. Step-indexed timeline — `gantt`, `gantt_html`
6. Per-node detail — `code_log`, `error_summary`, `message_stream`, `diff_system_prompts`
7. Cost & reports — `token_sparkline`, `budget_burndown`, `report_md`
8. Comparison across runs — `bench_table`
9. CLI equivalents

Full roadmap: [`docs/internal/visualization_roadmap.md`](../../docs/internal/visualization_roadmap.md).


## 1. Load a trace

`load_trace` returns a `Trace` with `states` (one snapshot per step) and `metadata`. Everything below is a function over `states`.

In [1]:
from rlmflow.utils.trace import load_trace

trace = load_trace("../data/notebook-coding-agent/trace")
states = trace.states
print(f"loaded {len(states)} states  ·  metadata: {trace.metadata or '(none)'}")

loaded 5 states  ·  metadata: {'task': 'Create a simple boids simulation in plain html and javascript. Render 100s of fast moving and coloful boids. Do not add configurations -- just the canvas. The speed should be a constant, not based on size. Split each component into separate files and delegate the work. After the delegation finishes, verify the components. The main runnable interface should be in index.html. Save in output/boids-simulation.'}


## 2. Inline tree rendering

Every `Node` defines `_repr_html_` and `_repr_mimebundle_`, so just returning a node from a Jupyter cell renders it as a styled tree. Trees can collapse on the final `ResultNode`, so we pick the *richest* state for a complete picture.

In [2]:
richest = max(states, key=lambda s: len(s.walk()))
richest

root [query] {default}
  root [action] {default:gpt-5}
    root [supervising] {default:gpt-5}
      root.index_html [query] {fast}
        root.index_html [action] {fast:gpt-5-mini}
          root.index_html [result] {fast:gpt-5-mini} -> OK
      root.main_js [query] {fast}
        root.main_js [action] {fast:gpt-5-mini}
          root.main_js [result] {fast:gpt-5-mini} -> OK
      root.boid_js [query] {fast}
        root.boid_js [action] {fast:gpt-5-mini}
          root.boid_js [result] {fast:gpt-5-mini} -> OK
      root.sim_js [query] {fast}
        root.sim_js [action] {fast:gpt-5-mini}
          root.sim_js [result] {fast:gpt-5-mini} -> OK
      root [resume] {default:gpt-5}
        root [action] {default:gpt-5}
          root [result] {default:gpt-5} -> Wrote and verified output/boids-simulation/{index.html, main.js, sim.js, boid.js

In [3]:
print(richest.tree())

root [query] {default}
  root [action] {default:gpt-5}
    root [supervising] {default:gpt-5}
      root.index_html [query] {fast}
        root.index_html [action] {fast:gpt-5-mini}
          root.index_html [result] {fast:gpt-5-mini} -> OK
      root.main_js [query] {fast}
        root.main_js [action] {fast:gpt-5-mini}
          root.main_js [result] {fast:gpt-5-mini} -> OK
      root.boid_js [query] {fast}
        root.boid_js [action] {fast:gpt-5-mini}
          root.boid_js [result] {fast:gpt-5-mini} -> OK
      root.sim_js [query] {fast}
        root.sim_js [action] {fast:gpt-5-mini}
          root.sim_js [result] {fast:gpt-5-mini} -> OK
      root [resume] {default:gpt-5}
        root [action] {default:gpt-5}
          root [result] {default:gpt-5} -> Wrote and verified output/boids-simulation/{index.html, main.js, sim.js, boid.js


In [4]:
from rlmflow.utils.viz import ascii_boxes

print(ascii_boxes(richest))

╭────────────────────────────────────────────── root  [query]  {default} ──────────────────────────────────────────────╮
│                                                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
└── ╭──────────────────────────────────────── root  [action]  {default:gpt-5} ─────────────────────────────────────────╮
    │                                                                                                                  │
    ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
    └── ╭──────────────────────────────────── root  [supervising]  {default:gpt-5} ────────────────────────────────────╮
        │                                                                                                              │
        ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
        ├── ╭──────────────────────────────────── root.index_html  [query]  {fast} ────────────────────────────────────╮
        │   │                                                                                                          │
        │   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────╯
        │   └── ╭──────────────────────────── root.index_html  [action]  {fast:gpt-5-mini} ────────────────────────────╮
        │       │                                                                                                      │
        │       ╰──────────────────────────────────────────────────────────────────────────────────────────────────────╯
        │       └── ╭────────────────────────── root.index_html  [result]  {fast:gpt-5-mini} ──────────────────────────╮
        │           │ OK                                                                                               │
        │           ╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
        ├── ╭───────────────────────────────────── root.main_js  [query]  {fast} ──────────────────────────────────────╮
        │   │                                                                                                          │
        │   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────╯
        │   └── ╭───────────────────────────── root.main_js  [action]  {fast:gpt-5-mini} ──────────────────────────────╮
        │       │                                                                                                      │
        │       ╰──────────────────────────────────────────────────────────────────────────────────────────────────────╯
        │       └── ╭─────────────────────────── root.main_js  [result]  {fast:gpt-5-mini} ────────────────────────────╮
        │           │ OK                                                                                               │
        │           ╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
        ├── ╭───────────────────────────────────── root.boid_js  [query]  {fast} ──────────────────────────────────────╮
        │   │                                                                                                          │
        │   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────╯
        │   └── ╭───────────────────────────── root.boid_js  [action]  {fast:gpt-5-mini} ──────────────────────────────╮
        │       │                                                                                                      │
        │       ╰──────────────────────────────────────────────────────────────────────────────────────────────────────╯
       

╭────────────────────────────────────────────── root  [query]  {default} ──────────────────────────────────────────────╮
│                                                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
└── ╭──────────────────────────────────────── root  [action]  {default:gpt-5} ─────────────────────────────────────────╮
    │                                                                                                                  │
    ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
    └── ╭──────────────────────────────────── root  [supervising]  {default:gpt-5} ────────────────────────────────────╮
        │                                                                                                              │
        ╰───────────────────────

## 3. Interactive viewer

The full interactive graph + per-agent chat lives in `open_viewer` (see section just below). It embeds inline in Jupyter via Gradio, with a step slider, clickable nodes, and a color-coded conversation panel for any agent you click.

For static export (PR comments, email), use the topology renders in section 4 (`mermaid`, `dot`, `d2`) or `rlmflow render <path> -f gantt-html -o gantt.html`.

### Inline Gradio viewer — clickable tree + per-agent chat

`open_viewer` ships an interactive viewer that embeds inline in Jupyter via Gradio's `inline=True` launch flag. The layout:

- **Left** — a clickable tree of every node in the run (one row per node, indented by depth, marker per type — `▢ query`, `▸ action`, `○ observation`, `↻ resume`, `• supervising`, `✓ result`, `✗ error`).
- **Right** — the **full conversation** for the agent that owns the clicked node: `system` prompt, the original `user` query, every `assistant` reply (including the REPL code blocks), every `tool` observation, and the final `done()` result.
- **Bottom** — collapsed accordion with the picked node's summary and raw JSON.

Pass `session=` (or a path to a `session/` dir) to get the *full* per-agent message history. With only `states`, you only get whatever the snapshots captured (often shallow — see how thin the conversation is for a query-only node).

The cell starts a local Gradio server in the background; call `gradio.close_all()` to stop it.

In [5]:
len(states[1].walk())

7

In [6]:
from rlmflow.utils.viewer import open_viewer

# Pass `session=` so the viewer can reconstruct the full per-agent chat
# (system / user / assistant / tool / result), not just the snapshot tree.
# `states` alone gives you only what was captured in the trace.
open_viewer(
    states,
    session="../data/notebook-coding-agent/session",
    inline=True,
    height=720,
    quiet=True,
)

## 4. Topology renders

Static topology renders for embedding in docs, PRs, post-mortems. Mermaid blocks render inline via `IPython.display.Markdown`. The same renders are reachable from the CLI as `rlmflow render <path> -f F`.

In [7]:
from IPython.display import Markdown
from rlmflow.utils.export import to_mermaid

Markdown(f"```mermaid\n{to_mermaid(richest)}\n```")

```mermaid
stateDiagram-v2
    state "root (query)" as n_ee015a8407fa
    state "root (action)" as n_4ee09ca8eafb
    state "root (supervising)" as n_a982d205ccd6
    state "root.index_html (query)" as n_6bb7c66b5fc3
    state "root.index_html (action)" as n_20e080a761b1
    state "root.index_html (result)" as n_6b71432a9d4f
    state "root.main_js (query)" as n_1c0cc3aae3c2
    state "root.main_js (action)" as n_3a9aee2c62a3
    state "root.main_js (result)" as n_f8af16869c1d
    state "root.boid_js (query)" as n_bf2e81f1e4b5
    state "root.boid_js (action)" as n_1e0483326c2d
    state "root.boid_js (result)" as n_71a2c960c332
    state "root.sim_js (query)" as n_9226b829d525
    state "root.sim_js (action)" as n_5006e0d65a48
    state "root.sim_js (result)" as n_f4632c0cfd5e
    state "root (resume)" as n_5db2399a7849
    state "root (action)" as n_71503518492b
    state "root (result)" as n_aeb10806c0f1
    [*] --> n_ee015a8407fa
    n_6b71432a9d4f --> [*] : OK
    n_20e080a761b1 --> n_6b71432a9d4f
    n_6bb7c66b5fc3 --> n_20e080a761b1
    n_a982d205ccd6 --> n_6bb7c66b5fc3
    n_f8af16869c1d --> [*] : OK
    n_3a9aee2c62a3 --> n_f8af16869c1d
    n_1c0cc3aae3c2 --> n_3a9aee2c62a3
    n_a982d205ccd6 --> n_1c0cc3aae3c2
    n_71a2c960c332 --> [*] : OK
    n_1e0483326c2d --> n_71a2c960c332
    n_bf2e81f1e4b5 --> n_1e0483326c2d
    n_a982d205ccd6 --> n_bf2e81f1e4b5
    n_f4632c0cfd5e --> [*] : OK
    n_5006e0d65a48 --> n_f4632c0cfd5e
    n_9226b829d525 --> n_5006e0d65a48
    n_a982d205ccd6 --> n_9226b829d525
    n_aeb10806c0f1 --> [*] : Wrote and verified output/boids-simulation/{index.html, mai...
    n_71503518492b --> n_aeb10806c0f1
    n_5db2399a7849 --> n_71503518492b
    n_a982d205ccd6 --> n_5db2399a7849
    n_4ee09ca8eafb --> n_a982d205ccd6
    n_ee015a8407fa --> n_4ee09ca8eafb
```

In [8]:
from rlmflow.utils.export import to_mermaid_flowchart

Markdown(f"```mermaid\n{to_mermaid_flowchart(richest)}\n```")

```mermaid
flowchart TD
    n_ee015a8407fa["root<br/><i>query</i>"]:::query
    n_ee015a8407fa --> n_4ee09ca8eafb
    n_4ee09ca8eafb["root<br/><i>action</i>"]:::action
    n_4ee09ca8eafb --> n_a982d205ccd6
    n_a982d205ccd6["root<br/><i>supervising</i>"]:::sup
    n_a982d205ccd6 --> n_6bb7c66b5fc3
    n_6bb7c66b5fc3["root.index_html<br/><i>query</i>"]:::query
    n_6bb7c66b5fc3 --> n_20e080a761b1
    n_20e080a761b1["root.index_html<br/><i>action</i>"]:::action
    n_20e080a761b1 --> n_6b71432a9d4f
    n_6b71432a9d4f["root.index_html<br/><i>result</i><br/>OK"]:::result
    n_a982d205ccd6 --> n_1c0cc3aae3c2
    n_1c0cc3aae3c2["root.main_js<br/><i>query</i>"]:::query
    n_1c0cc3aae3c2 --> n_3a9aee2c62a3
    n_3a9aee2c62a3["root.main_js<br/><i>action</i>"]:::action
    n_3a9aee2c62a3 --> n_f8af16869c1d
    n_f8af16869c1d["root.main_js<br/><i>result</i><br/>OK"]:::result
    n_a982d205ccd6 --> n_bf2e81f1e4b5
    n_bf2e81f1e4b5["root.boid_js<br/><i>query</i>"]:::query
    n_bf2e81f1e4b5 --> n_1e0483326c2d
    n_1e0483326c2d["root.boid_js<br/><i>action</i>"]:::action
    n_1e0483326c2d --> n_71a2c960c332
    n_71a2c960c332["root.boid_js<br/><i>result</i><br/>OK"]:::result
    n_a982d205ccd6 --> n_9226b829d525
    n_9226b829d525["root.sim_js<br/><i>query</i>"]:::query
    n_9226b829d525 --> n_5006e0d65a48
    n_5006e0d65a48["root.sim_js<br/><i>action</i>"]:::action
    n_5006e0d65a48 --> n_f4632c0cfd5e
    n_f4632c0cfd5e["root.sim_js<br/><i>result</i><br/>OK"]:::result
    n_a982d205ccd6 --> n_5db2399a7849
    n_5db2399a7849["root<br/><i>resume</i>"]:::resume
    n_5db2399a7849 --> n_71503518492b
    n_71503518492b["root<br/><i>action</i>"]:::action
    n_71503518492b --> n_aeb10806c0f1
    n_aeb10806c0f1["root<br/><i>result</i><br/>Wrote and verified output/boids-simulat..."]:::result
    classDef query    fill:#1f6feb22,stroke:#58a6ff,color:#c9d1d9;
    classDef obs      fill:#1f6feb22,stroke:#58a6ff,color:#c9d1d9;
    classDef action   fill:#d2992222,stroke:#d29922,color:#c9d1d9;
    classDef sup      fill:#bc8cff22,stroke:#bc8cff,color:#c9d1d9;
    classDef resume   fill:#7ee78722,stroke:#7ee787,color:#c9d1d9;
    classDef err      fill:#f8514922,stroke:#f85149,color:#c9d1d9;
    classDef result   fill:#3fb95022,stroke:#3fb950,color:#c9d1d9;
```

In [9]:
from rlmflow.utils.export import to_mermaid_sequence

Markdown(f"```mermaid\n{to_mermaid_sequence(richest)}\n```")

```mermaid
sequenceDiagram
    participant root as root
    participant root_index_html as root.index_html
    participant root_main_js as root.main_js
    participant root_boid_js as root.boid_js
    participant root_sim_js as root.sim_js
    root->>+root_index_html: delegate
    root_index_html-->>-root: OK
    root->>+root_main_js: delegate
    root_main_js-->>-root: OK
    root->>+root_boid_js: delegate
    root_boid_js-->>-root: OK
    root->>+root_sim_js: delegate
    root_sim_js-->>-root: OK
```

In [10]:
from rlmflow.utils.export import to_dot, to_d2

print("--- DOT (paste into a Graphviz renderer) ---")
print(to_dot(richest))
print()
print("--- D2 (https://d2lang.com) ---")
print(to_d2(richest))

--- DOT (paste into a Graphviz renderer) ---
digraph rlmflow {
    rankdir=TB;
    node [shape=box, style="rounded,filled", fontname="Helvetica"];
    edge [fontname="Helvetica", fontsize=10];
    n_ee015a8407fa [label="root\nquery", fillcolor="#58a6ff22", color="#58a6ff"];
    n_ee015a8407fa -> n_4ee09ca8eafb;
    n_4ee09ca8eafb [label="root\naction", fillcolor="#d2992222", color="#d29922"];
    n_4ee09ca8eafb -> n_a982d205ccd6;
    n_a982d205ccd6 [label="root\nsupervising", fillcolor="#bc8cff22", color="#bc8cff"];
    n_a982d205ccd6 -> n_6bb7c66b5fc3;
    n_6bb7c66b5fc3 [label="root.index_html\nquery", fillcolor="#58a6ff22", color="#58a6ff"];
    n_6bb7c66b5fc3 -> n_20e080a761b1;
    n_20e080a761b1 [label="root.index_html\naction", fillcolor="#d2992222", color="#d29922"];
    n_20e080a761b1 -> n_6b71432a9d4f;
    n_6b71432a9d4f [label="root.index_html\nresult\nOK", fillcolor="#3fb95022", color="#3fb950"];
    n_a982d205ccd6 -> n_1c0cc3aae3c2;
    n_1c0cc3aae3c2 [label="root.main_js\n

## 5. Step-indexed timeline

`gantt` prints a Rich table — one row per agent, one column per step, single-letter glyphs (`Q`/`A`/`O`/`S`/`R`/`F`/`E`). `gantt_html` writes a colorful self-contained HTML page; we render it inline below.

In [11]:
from rlmflow.utils.viz import gantt

gantt(states)

┏━━━━━━━━━━━━━━━┳━┳━┳━┳━┳━┓
┃agent          ┃0┃1┃2┃3┃4┃
┡━━━━━━━━━━━━━━━╇━╇━╇━╇━╇━┩
│root           │Q│S│S│R│F│
│root.index_html│ │Q│F│F│F│
│root.main_js   │ │Q│F│F│F│
│root.boid_js   │ │Q│F│F│F│
│root.sim_js    │ │Q│F│F│F│
└───────────────┴─┴─┴─┴─┴─┘

In [12]:
from IPython.display import HTML
from rlmflow.utils.viz import gantt_html

HTML(gantt_html(states, title="boids run"))

## 6. Per-node detail

Drill into the actual code that ran, errors that happened, and the chat-log view for any agent.

In [13]:
from rlmflow.utils.viz import code_log

print(code_log(states))

# [root] action
# Block 1: Delegate writing each component file with exact content, then wait. No done() here.

index_html = """<!doctype html>
<html lang="en">
  <head>
    <meta charset="utf-8" />
    <title>Boids Simulation</title>
    <meta name="viewport" content="width=device-width, initial-scale=1" />
    <style>
      html, body {
        margin: 0;
        height: 100%;
        background: #000;
        overflow: hidden;
      }
      canvas {
        display: block;
        width: 100vw;
        height: 100vh;
      }
    </style>
  </head>
  <body>
    <canvas id="c"></canvas>
    <script type="module" src="./main.js"></script>
  </body>
</html>
"""

main_js = """import { Simulation } from './sim.js';

const canvas = document.getElementById('c');

function resize() {
  const dpr = Math.max(1, window.devicePixelRatio || 1);
  const w = window.innerWidth;
  const h = window.innerHeight;
  canvas.style.width = w + 'px';
  canvas.style.height = h + 'px';
  canvas.width = Math.fl

In [14]:
# The boids run had no errors — synthesize a tiny example so error_summary has something to group.
from rlmflow import ErrorNode, QueryNode
from rlmflow.utils.viz import error_summary

demo = QueryNode(
    agent_id="root",
    children=[
        ErrorNode(agent_id="root.a", error="no_code_block", content="missing repl block"),
        ErrorNode(agent_id="root.b", error="no_code_block", content="empty reply"),
        ErrorNode(agent_id="root.c", error="orphaned_delegates", content="never waited"),
    ],
)
print(error_summary(demo))

3 error(s) across 2 kind(s):
  no_code_block: 2
    └─ missing repl block
  orphaned_delegates: 1
    └─ never waited


In [15]:
from pathlib import Path
from rlmflow.workspace import FileSession
from rlmflow.utils.viz import message_stream

session = FileSession(Path("../data/notebook-coding-agent/session"))
stream = message_stream("root.index_html", session)
print(stream[:1800] + ("\n..." if len(stream) > 1800 else ""))

--- system ---
## Role

You are a recursive agent with a Python REPL. You solve tasks by writing and executing Python programs, and you can delegate subtasks to sub-agents with fresh context windows.

## REPL

- Every response is exactly one ```repl``` code block. Tools are already in the namespace.
- Variables persist across turns within one agent.
- `AGENT_ID`, `DEPTH`, `MAX_DEPTH` are set; cannot `delegate` when `DEPTH == MAX_DEPTH`.
- **Final answer:** call `done(answer)` exactly once when complete — that string is what the parent/user sees. No `done`, no result.
- **End the block after `wait`. Verify on the next turn.** The runtime won't stop you — if you call `done()` in the same block, it ends the agent right there with no verify turn. Instead, after `yield wait(...)` resumes, *return without calling `done()`*. The runtime then gives you a fresh turn (observation: `Children finished: ... / Generator resumed. Output: ...`) where you read files back / run / grep the artifact, and 

In [16]:
from rlmflow.utils.viz import diff_system_prompts

by_agent = {n.agent_id: n for n in session.load().values()}
children = sorted(aid for aid in by_agent if aid != "root")
print("child agents:", children)
a, b = children[0], children[1]
print(f"\ndiffing: {a}  vs  {b}\n")
diff = diff_system_prompts(by_agent[a], by_agent[b])
print(diff[:1500] if diff != "(prompts identical)" else diff)

child agents: ['root.boid_js', 'root.index_html', 'root.main_js', 'root.sim_js']

diffing: root.boid_js  vs  root.index_html

(prompts identical)


## 7. Cost & reports

Three views: a one-line ASCII sparkline of cumulative tokens, a budget burndown bar, and a full Markdown report bundling tree + cost + result + errors.

In [17]:
from rlmflow.utils.viz import token_sparkline, budget_burndown

print(token_sparkline(states))
print(budget_burndown(states))
print(budget_burndown(states, max_budget=50_000))

 ▁▄▅█  103371 tok over 5 steps
[████████████████████████████████████████] 100.0%  103371 tok (peak)
[████████████████████████████████████████] 100.0%  103371/50000


In [18]:
from rlmflow.utils.viz import report_md

Markdown(report_md(states, title="boids — coding-agent run"))

# boids — coding-agent run

**Steps:** 5
**Agents:** 5
**Tokens:** 103,371 (54,834 in, 48,537 out)
**Outcome:** result

## Tree

```
root [query] {default}
  root [action] {default:gpt-5}
    root [supervising] {default:gpt-5}
      root.index_html [query] {fast}
        root.index_html [action] {fast:gpt-5-mini}
          root.index_html [result] {fast:gpt-5-mini} -> OK
      root.main_js [query] {fast}
        root.main_js [action] {fast:gpt-5-mini}
          root.main_js [result] {fast:gpt-5-mini} -> OK
      root.boid_js [query] {fast}
        root.boid_js [action] {fast:gpt-5-mini}
          root.boid_js [result] {fast:gpt-5-mini} -> OK
      root.sim_js [query] {fast}
        root.sim_js [action] {fast:gpt-5-mini}
          root.sim_js [result] {fast:gpt-5-mini} -> OK
      root [resume] {default:gpt-5}
        root [action] {default:gpt-5}
          root [result] {default:gpt-5} -> Wrote and verified output/boids-simulation/{index.html, main.js, sim.js, boid.js
```

## Cumulative tokens

```
 ▁▄▅█  103371 tok over 5 steps
```

## Result

```
Wrote and verified output/boids-simulation/{index.html, main.js, sim.js, boid.js}. Open output/boids-simulation/index.html to run the simulation.
```


## 8. Comparison across runs

`bench_table` aggregates labeled traces — use it to compare benchmark runs side by side (cost, duration, outcome, errors).

In [19]:
from rlmflow.utils.viz import bench_table

# Fake two runs by trimming the same trace, just to show the table shape.
print(bench_table({
    "boids:full":      states,
    "boids:firsthalf": states[: len(states) // 2 + 1],
}))

label            steps  agents  outcome  tokens   errors
---------------  -----  ------  -------  -------  ------
boids:full       5      5       open     103,371  0     
boids:firsthalf  3      5       open     56,314   0     


## 9. CLI equivalents

Every render here is also available without writing Python. From a shell:

```bash
rlmflow view   examples/data/notebook-coding-agent/trace
rlmflow render examples/data/notebook-coding-agent/trace -f mermaid-flowchart
rlmflow render examples/data/notebook-coding-agent/trace -f gantt-html  -o gantt.html
rlmflow render examples/data/notebook-coding-agent/trace -f report-md   -o report.md
rlmflow render examples/data/notebook-coding-agent/trace -f code-log
rlmflow render examples/data/notebook-coding-agent/trace -f tokens
```

All formats: `mermaid` · `mermaid-flowchart` · `mermaid-sequence` · `dot` · `d2` · `tree` · `ascii-boxes` · `gantt-html` · `report-md` · `code-log` · `error-summary` · `tokens`.